# Vessel Segmentation Training and Testing
## Blood Vessel Segmentation using LadderNet/U-Net

This notebook integrates training and testing for retinal vessel segmentation on DRIVE dataset.

## 1. Environment Setup (Colab)

Upload your code to Google Drive or clone from GitHub, then run this cell to setup environment.

In [ ]:
# Mount Google Drive (if using Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print('Running in Google Colab')
except:
    IN_COLAB = False
    print('Running locally')

# Change to project directory
import os
if IN_COLAB:
    # Modify this path to your project location in Google Drive
    project_path = '/content/drive/MyDrive/VesselSeg-Pytorch'
    if not os.path.exists(project_path):
        print(f'⚠️  Project path not found: {project_path}')
        print('Please upload your project to Google Drive or clone from GitHub')
    else:
        os.chdir(project_path)
        print(f'✓ Changed to project directory: {project_path}')
else:
    print(f'Current directory: {os.getcwd()}')

In [ ]:
# Install required packages
!pip install -q opencv-python-headless tensorboardX pylibtiff protobuf==3.20.3 joblib

# Verify GPU availability
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB')

## 2. Import Libraries and Modules

## 2. Verify Project Structure (Optional)

Run this cell to check if all required files exist.

In [ ]:
import os

# Check if required files and folders exist
required_items = [
    'lib/',
    'models/',
    'config.py',
    'function.py',
    'test.py',
    'train.py',
    'prepare_dataset/data_path_list/DRIVE/train.txt',
    'prepare_dataset/data_path_list/DRIVE/test.txt'
]

print('Checking project structure...')
print(f'Current directory: {os.getcwd()}\n')

all_exist = True
for item in required_items:
    exists = os.path.exists(item)
    status = '✓' if exists else '✗'
    print(f'{status} {item}')
    if not exists:
        all_exist = False

if all_exist:
    print('\n✅ All required files found!')
else:
    print('\n⚠️  Some files are missing. Please check your project structure.')
    print('\nTip: Make sure you have uploaded the entire project folder to Colab/Jupyter.')

## 3. Import Libraries and Modules

In [ ]:
import sys
import os
from os.path import join

# Add project root to Python path (must be first)
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
import torch.backends.cudnn as cudnn
import torch.optim as optim
import time
import numpy as np
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

# Import project modules
from lib.losses.loss import CrossEntropyLoss2d
from lib.common import setpu_seed, save_args, count_parameters
from config import parse_args
from lib.logger import Logger, Print_Logger
import models
from function import get_dataloaderV2, train, val

# Import Test class from local test.py (avoid conflict with stdlib test module)
import importlib.util
spec = importlib.util.spec_from_file_location("test_module", os.path.join(project_root, "test.py"))
test_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(test_module)
Test = test_module.Test

# Enable TensorBoard in notebook
%load_ext tensorboard

print('✓ All modules imported successfully!')

## 4. Configuration

Set training hyperparameters and paths.

In [ ]:
# Create a custom args object for Jupyter
class Args:
    def __init__(self):
        # Paths
        self.outf = './experiments'
        self.save = 'UNet_vessel_seg_colab'
        self.train_data_path_list = './prepare_dataset/data_path_list/DRIVE/train.txt'
        self.test_data_path_list = './prepare_dataset/data_path_list/DRIVE/test.txt'
        
        # Data parameters
        self.train_patch_height = 64
        self.train_patch_width = 64
        self.N_patches = 150000
        self.inside_FOV = 'center'
        self.val_ratio = 0.1
        self.sample_visualization = True
        
        # Model parameters
        self.in_channels = 1
        self.classes = 2
        
        # Training parameters
        self.N_epochs = 50
        self.batch_size = 64
        self.early_stop = 6
        self.lr = 0.0005
        self.val_on_test = False
        
        # Pre-trained
        self.start_epoch = 1
        self.pre_trained = None
        
        # Testing parameters
        self.test_patch_height = 96
        self.test_patch_width = 96
        self.stride_height = 16
        self.stride_width = 16
        
        # Hardware
        self.cuda = True

args = Args()

# Display configuration
print('Configuration:')
print(f'  Experiment name: {args.save}')
print(f'  Epochs: {args.N_epochs}')
print(f'  Batch size: {args.batch_size}')
print(f'  Learning rate: {args.lr}')
print(f'  Patch size: {args.train_patch_height}x{args.train_patch_width}')
print(f'  Number of patches: {args.N_patches}')
print(f'  Early stopping: {args.early_stop} epochs')

## 5. Setup Training Environment

In [ ]:
# Set random seed for reproducibility
setpu_seed(2021)

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() and args.cuda else "cpu")
cudnn.benchmark = True
print(f'Computing device: {"GPU" if device.type=="cuda" else "CPU"}')

# Create save directory
save_path = join(args.outf, args.save)
os.makedirs(save_path, exist_ok=True)
save_args(args, save_path)
print(f'Save path: {save_path}')

# Initialize logger
log = Logger(save_path)
print('✓ Environment setup complete!')

## 6. Build Model

In [ ]:
# Initialize model
# Option 1: LadderNet (default)
net = models.LadderNet(inplanes=args.in_channels, num_classes=args.classes, 
                       layers=3, filters=16).to(device)

# Option 2: U-Net (uncomment to use)
# net = models.UNetFamily.U_Net(args.in_channels, args.classes).to(device)

print(f"Total number of parameters: {count_parameters(net):,}")

# Save model architecture to tensorboard
log.save_graph(net, torch.randn((1, 1, 48, 48)).to(device))

# Initialize loss and optimizer
criterion = CrossEntropyLoss2d()
optimizer = optim.Adam(net.parameters(), lr=args.lr)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.N_epochs, eta_min=0)

print('✓ Model built successfully!')

## 7. Load Data

In [ ]:
# Create dataloaders
print('Loading datasets...')
train_loader, val_loader = get_dataloaderV2(args)

print(f'Training samples: {len(train_loader.dataset)}')
print(f'Validation samples: {len(val_loader.dataset)}')
print(f'Batches per epoch: {len(train_loader)}')

# Initialize validation tool (optional)
if args.val_on_test:
    print('\033[0;32m===============Validation on Testset!!!===============\033[0m')
    val_tool = Test(args)

print('✓ Data loaded successfully!')

## 8. Training Loop

Train the model with real-time visualization.

In [ ]:
# Training history for plotting
history = {
    'train_loss': [],
    'val_loss': [],
    'val_auc': [],
    'val_acc': []
}

best = {'epoch': 0, 'AUC_roc': 0.5}
trigger = 0

# Training loop
for epoch in range(args.start_epoch, args.N_epochs + 1):
    print(f'\n{"="*60}')
    print(f'EPOCH: {epoch}/{args.N_epochs} -- LR: {optimizer.state_dict()["param_groups"][0]["lr"]:.6f}')
    print(f'Time: {time.asctime()}')
    print(f'{"="*60}')
    
    # Train stage
    train_log = train(train_loader, net, criterion, optimizer, device)
    
    # Validation stage
    if not args.val_on_test:
        val_log = val(val_loader, net, criterion, device)
    else:
        val_tool.inference(net)
        val_log = val_tool.val()
    
    # Log results
    log.update(epoch, train_log, val_log)
    lr_scheduler.step()
    
    # Store history
    history['train_loss'].append(train_log['train_loss'])
    history['val_loss'].append(val_log.get('val_loss', 0))
    history['val_auc'].append(val_log['val_auc_roc'])
    history['val_acc'].append(val_log['val_acc'])
    
    # Save checkpoint
    state = {'net': net.state_dict(), 'optimizer': optimizer.state_dict(), 'epoch': epoch}
    torch.save(state, join(save_path, 'latest_model.pth'))
    
    trigger += 1
    if val_log['val_auc_roc'] > best['AUC_roc']:
        print('\033[0;33m💾 Saving best model!\033[0m')
        torch.save(state, join(save_path, 'best_model.pth'))
        best['epoch'] = epoch
        best['AUC_roc'] = val_log['val_auc_roc']
        trigger = 0
    
    print(f'\n🏆 Best performance at Epoch: {best["epoch"]} | AUC_roc: {best["AUC_roc"]:.4f}')
    
    # Plot training progress
    if epoch % 5 == 0 or epoch == 1:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        
        # Plot loss
        axes[0].plot(history['train_loss'], label='Train Loss')
        if history['val_loss'][0] != 0:
            axes[0].plot(history['val_loss'], label='Val Loss')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].set_title('Training & Validation Loss')
        axes[0].legend()
        axes[0].grid(True)
        
        # Plot AUC
        axes[1].plot(history['val_auc'], label='Val AUC-ROC', color='green')
        axes[1].axhline(y=best['AUC_roc'], color='r', linestyle='--', label=f"Best: {best['AUC_roc']:.4f}")
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('AUC-ROC')
        axes[1].set_title('Validation AUC-ROC')
        axes[1].legend()
        axes[1].grid(True)
        
        # Plot Accuracy
        axes[2].plot(history['val_acc'], label='Val Accuracy', color='orange')
        axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('Accuracy')
        axes[2].set_title('Validation Accuracy')
        axes[2].legend()
        axes[2].grid(True)
        
        plt.tight_layout()
        plt.savefig(join(save_path, 'training_curves.png'), dpi=150, bbox_inches='tight')
        plt.show()
    
    # Early stopping
    if args.early_stop is not None:
        if trigger >= args.early_stop:
            print(f"\n⚠️  Early stopping triggered at epoch {epoch}")
            break
    
    torch.cuda.empty_cache()

print(f'\n{"="*60}')
print('✅ Training completed!')
print(f'🏆 Best model saved at epoch {best["epoch"]} with AUC: {best["AUC_roc"]:.4f}')
print(f'{"="*60}')

## 8. View TensorBoard

Monitor training progress in TensorBoard.

In [ ]:
# Launch TensorBoard
%tensorboard --logdir {save_path}

## 9. Testing & Evaluation

Load the best model and evaluate on test set.

In [ ]:
# Load best model
print('Loading best model...')
checkpoint = torch.load(join(save_path, 'best_model.pth'))
net.load_state_dict(checkpoint['net'])
print(f'✓ Loaded model from epoch {checkpoint["epoch"]}')

# Initialize test tool
eval_tool = Test(args)

# Run inference
print('\nRunning inference on test set...')
eval_tool.inference(net)

# Evaluate and print results
print('\nEvaluating results...')
test_results = eval_tool.evaluate()

print('\n' + '='*60)
print('📊 Test Results:')
print('='*60)
for key, value in test_results.items():
    print(f'{key:20s}: {value:.6f}')
print('='*60)

# Save segmentation results
print('\nSaving segmentation results...')
eval_tool.save_segmentation_result()
print(f'✓ Results saved to: {eval_tool.save_img_path}')

## 10. Visualize Results

Display some segmentation results.

In [ ]:
import cv2
from glob import glob

# Get result images
result_imgs = sorted(glob(join(eval_tool.save_img_path, 'Result_*.png')))

# Display first few results
num_display = min(4, len(result_imgs))
fig, axes = plt.subplots(num_display, 1, figsize=(15, 4*num_display))

if num_display == 1:
    axes = [axes]

for i, img_path in enumerate(result_imgs[:num_display]):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    axes[i].set_title(os.path.basename(img_path), fontsize=12)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig(join(save_path, 'visualization_summary.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'\n✓ Visualized {num_display} test results')
print(f'Total test images: {len(result_imgs)}')

## 11. Download Results (Colab only)

Download the experiment folder with all results.

In [ ]:
if IN_COLAB:
    # Create zip file
    import shutil
    zip_path = f'/content/{args.save}'
    shutil.make_archive(zip_path, 'zip', save_path)
    
    # Download
    from google.colab import files
    files.download(f'{zip_path}.zip')
    print(f'✓ Downloaded: {args.save}.zip')
else:
    print(f'Results saved at: {save_path}')

## Summary

This notebook provides:
- ✅ Complete training pipeline calling your existing .py modules
- ✅ Real-time training visualization
- ✅ TensorBoard integration
- ✅ Automatic model checkpointing
- ✅ Test set evaluation
- ✅ Results visualization
- ✅ Google Colab compatibility

### Key Features:
1. **Imports all your existing functions** from `train.py`, `test.py`, `function.py`, etc.
2. **No code duplication** - reuses your existing codebase
3. **Interactive visualization** - plots update during training
4. **Easy to customize** - modify hyperparameters in the Args class
5. **Colab-ready** - works with Google Drive mounting